# Movement Type Detection — 模型训练完整流水线

**目标**: 基于 IMU 6轴数据（加速度计 + 陀螺仪），训练一个轻量级 CNN 分类模型，区分 3 种运动类型：
- `stationary` (静止)
- `shaking` (抖动)
- `circle` (画圈)

**输出**: 量化后的 TFLite 模型，转为 `model.h` / `model.c`（C 字节数组），可部署到 PSoC-Edge 等嵌入式开发板。

**参数设定**:
- 采样率: 50 Hz
- 窗口大小: 2秒 (100 采样点)
- 窗口步进: 50% 重叠 (50 采样点)
- 数据切分: 80/10/10 (train/val/test) 随机切分
- 量化: INT8 全量化
- 模型目标大小: < 100 KB

## 1. 环境准备与依赖安装

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'numpy', 'pandas', 'tensorflow', 'scikit-learn',
                       'matplotlib', 'seaborn'])
print('All dependencies installed.')

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 2. 全局参数配置

In [ ]:
# ============ 全局参数 ============
DATA_ROOT = os.path.join(os.getcwd(), 'Data')  # 数据根目录
SAMPLE_RATE = 50          # 采样率 Hz
WINDOW_SEC = 2.0          # 窗口时长 (秒)
WINDOW_SIZE = int(SAMPLE_RATE * WINDOW_SEC)   # 100 个采样点
STEP_SIZE = WINDOW_SIZE // 2                  # 50% 重叠 => 50
NUM_CHANNELS = 6          # Accel_X/Y/Z + Gyro_X/Y/Z

# 类别映射
LABEL_MAP = {'stationary': 0, 'shaking': 1, 'circle': 2}
LABEL_NAMES = ['stationary', 'shaking', 'circle']
NUM_CLASSES = len(LABEL_NAMES)

# 数据切分比例
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# 训练参数
BATCH_SIZE = 32
EPOCHS = 150
LEARNING_RATE = 5e-4

# 输出目录
OUTPUT_DIR = os.path.join(os.getcwd(), 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Window size: {WINDOW_SIZE} samples ({WINDOW_SEC}s)')
print(f'Step size:   {STEP_SIZE} samples ({STEP_SIZE/SAMPLE_RATE}s)')
print(f'Classes:     {LABEL_NAMES}')

## 3. 数据加载与标签解析

In [ ]:
def load_imu_data(filepath):
    """加载 IMU-Data.data 文件，返回 DataFrame"""
    df = pd.read_csv(filepath, comment='#',
                     names=['time', 'accel_x', 'accel_y', 'accel_z',
                            'gyro_x', 'gyro_y', 'gyro_z'])
    return df


def load_labels(filepath):
    """加载 Live-Labeling.label 文件，返回标签区间列表"""
    df = pd.read_csv(filepath)
    labels = []
    # 列名: Time(Seconds), Length(Seconds), Label(string), ...
    for _, row in df.iterrows():
        start = row.iloc[0]   # Time(Seconds)
        length = row.iloc[1]  # Length(Seconds)
        label = str(row.iloc[2]).strip().lower()  # Label(string)
        end = start + length
        labels.append((start, end, label))
    return labels


def assign_labels_to_samples(imu_df, label_intervals):
    """
    为每个 IMU 采样点分配标签。
    未被任何标签区间覆盖的 -> 'stationary'
    """
    times = imu_df['time'].values
    labels = np.full(len(times), 'stationary', dtype=object)
    
    for start, end, label in label_intervals:
        mask = (times >= start) & (times < end)
        labels[mask] = label
    
    return labels


print('Data loading functions defined.')

In [ ]:
def discover_sessions(data_root):
    """遍历 Data/ 下所有子文件夹，找到包含 IMU-Data.data 的 session"""
    sessions = []
    for person_dir in sorted(glob.glob(os.path.join(data_root, '*'))):
        if not os.path.isdir(person_dir):
            continue
        person_name = os.path.basename(person_dir)
        for session_dir in sorted(glob.glob(os.path.join(person_dir, '*'))):
            if not os.path.isdir(session_dir):
                continue
            imu_file = os.path.join(session_dir, 'IMU-Data.data')
            label_file = os.path.join(session_dir, 'Live-Labeling.label')
            if os.path.isfile(imu_file) and os.path.isfile(label_file):
                sessions.append({
                    'person': person_name,
                    'session': os.path.basename(session_dir),
                    'imu_path': imu_file,
                    'label_path': label_file
                })
    return sessions


sessions = discover_sessions(DATA_ROOT)
print(f'Found {len(sessions)} sessions total:')
for s in sessions[:5]:
    print(f"  {s['person']}/{s['session']}")
print('  ...')

## 4. 加载所有数据并生成逐样本标签

In [ ]:
all_segments = []  # 每个 session 的 (imu_array, label_array)

for s in sessions:
    imu_df = load_imu_data(s['imu_path'])
    label_intervals = load_labels(s['label_path'])
    sample_labels = assign_labels_to_samples(imu_df, label_intervals)
    
    # 提取 6 通道数值
    imu_values = imu_df[['accel_x', 'accel_y', 'accel_z',
                         'gyro_x', 'gyro_y', 'gyro_z']].values
    
    all_segments.append({
        'person': s['person'],
        'session': s['session'],
        'imu': imu_values,
        'labels': sample_labels,
        'n_samples': len(imu_values)
    })

# 统计信息
total_samples = sum(seg['n_samples'] for seg in all_segments)
total_duration = total_samples / SAMPLE_RATE
print(f'Loaded {len(all_segments)} sessions')
print(f'Total samples: {total_samples} ({total_duration:.1f} seconds / {total_duration/60:.1f} minutes)')

# 统计各类别总时长
label_counts = {'stationary': 0, 'shaking': 0, 'circle': 0}
for seg in all_segments:
    for lbl in LABEL_NAMES:
        label_counts[lbl] += np.sum(seg['labels'] == lbl)

print('\nPer-class sample counts (before windowing):')
for lbl, cnt in label_counts.items():
    print(f'  {lbl:12s}: {cnt:>8d} samples ({cnt/SAMPLE_RATE:.1f}s)')

## 5. 滑动窗口切分

In [ ]:
def create_windows(imu_data, labels, window_size, step_size, label_threshold=0.7):
    """
    从连续的 IMU 数据中创建滑动窗口。
    
    每个窗口的标签 = 该窗口中占比超过 label_threshold 的类别。
    如果没有类别超过阈值，则丢弃该窗口（过渡区域）。
    
    Returns:
        X: (N, window_size, num_channels)
        y: (N,) 整型标签
    """
    X_windows = []
    y_labels = []
    
    n_samples = len(imu_data)
    
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window_data = imu_data[start:end]
        window_labels = labels[start:end]
        
        # 统计窗口内各类别占比
        unique, counts = np.unique(window_labels, return_counts=True)
        proportions = dict(zip(unique, counts / window_size))
        
        # 找到占比最高的类别
        dominant_label = max(proportions, key=proportions.get)
        dominant_ratio = proportions[dominant_label]
        
        if dominant_ratio >= label_threshold:
            X_windows.append(window_data)
            y_labels.append(LABEL_MAP[dominant_label])
    
    return np.array(X_windows), np.array(y_labels)


# 对所有 session 执行窗口切分
X_all = []
y_all = []

for seg in all_segments:
    X_win, y_win = create_windows(seg['imu'], seg['labels'],
                                   WINDOW_SIZE, STEP_SIZE)
    if len(X_win) > 0:
        X_all.append(X_win)
        y_all.append(y_win)

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print(f'Total windows: {len(X_all)}')
print(f'X shape: {X_all.shape}  (windows, timesteps, channels)')
print(f'y shape: {y_all.shape}')
print()
for i, name in enumerate(LABEL_NAMES):
    print(f'  {name:12s}: {np.sum(y_all == i):>6d} windows')

## 6. 数据归一化

In [ ]:
# 计算全局 min/max（用于 INT8 量化和嵌入式端预处理）
# 在切分前用所有数据计算
channel_min = X_all.reshape(-1, NUM_CHANNELS).min(axis=0)
channel_max = X_all.reshape(-1, NUM_CHANNELS).max(axis=0)

print('Per-channel statistics (before normalization):')
ch_names = ['Accel_X', 'Accel_Y', 'Accel_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z']
for i, name in enumerate(ch_names):
    print(f'  {name:8s}:  min={channel_min[i]:>10.4f}  max={channel_max[i]:>10.4f}')

# Min-Max 归一化到 [0, 1]
X_normalized = (X_all - channel_min) / (channel_max - channel_min + 1e-8)

# 保存归一化参数（嵌入式端需要）
norm_params = {
    'channel_names': ch_names,
    'min': channel_min.tolist(),
    'max': channel_max.tolist()
}

print(f'\nNormalized X range: [{X_normalized.min():.4f}, {X_normalized.max():.4f}]')

## 7. 数据切分 (80/10/10)

In [ ]:
# 第一次切分: 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_normalized, y_all, test_size=0.2, random_state=42, stratify=y_all)

# 第二次切分: 从 temp 中 50/50 => 10% val, 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape[0]:>6d} windows ({X_train.shape[0]/len(X_normalized)*100:.1f}%)')
print(f'Val:   {X_val.shape[0]:>6d} windows ({X_val.shape[0]/len(X_normalized)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]:>6d} windows ({X_test.shape[0]/len(X_normalized)*100:.1f}%)')

print('\nPer-class distribution:')
for i, name in enumerate(LABEL_NAMES):
    tr = np.sum(y_train == i)
    va = np.sum(y_val == i)
    te = np.sum(y_test == i)
    print(f'  {name:12s}:  train={tr:>5d}  val={va:>5d}  test={te:>5d}')

## 8. 数据可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, split_name, y_split in zip(axes, ['Train', 'Validation', 'Test'],
                                    [y_train, y_val, y_test]):
    unique, counts = np.unique(y_split, return_counts=True)
    colors = ['#2ecc71', '#e74c3c', '#3498db']
    bars = ax.bar([LABEL_NAMES[u] for u in unique], counts, color=colors)
    ax.set_title(f'{split_name} Set')
    ax.set_ylabel('Window Count')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontsize=10)

plt.suptitle('Class Distribution Across Splits', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

In [ ]:
# 展示每种运动类型的样例窗口
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
time_axis = np.arange(WINDOW_SIZE) / SAMPLE_RATE

for i, (name, ax) in enumerate(zip(LABEL_NAMES, axes)):
    idx = np.where(y_train == i)[0][0]
    sample = X_train[idx]
    # 反归一化用于展示
    sample_raw = sample * (channel_max - channel_min) + channel_min
    
    for ch in range(3):
        ax.plot(time_axis, sample_raw[:, ch], label=ch_names[ch], linewidth=0.8)
    for ch in range(3, 6):
        ax.plot(time_axis, sample_raw[:, ch], label=ch_names[ch], linewidth=0.8, linestyle='--')
    
    ax.set_title(f'Sample: {name}', fontsize=12)
    ax.set_ylabel('Sensor Value')
    ax.legend(loc='upper right', ncol=3, fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (seconds)')
plt.suptitle('Sample Windows for Each Movement Type', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_windows.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. 类别平衡处理

In [ ]:
# 计算类别权重用于训练 (处理数据不平衡)
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weights = dict(enumerate(class_weights_array))

print('Class weights for balanced training:')
for i, name in enumerate(LABEL_NAMES):
    print(f'  {name:12s}: {class_weights[i]:.4f}')

## 10. 构建 Conv2D 模型

使用轻量级 Conv2D 架构，类似已有的 `conv2d-medium-balanced-2` 模型。
输入维度重塑为 `(100, 6, 1)` 以适配 Conv2D。

In [ ]:
# 为 Conv2D 增加通道维度: (N, 100, 6) -> (N, 100, 6, 1)
X_train_4d = X_train[..., np.newaxis]
X_val_4d = X_val[..., np.newaxis]
X_test_4d = X_test[..., np.newaxis]

print(f'X_train_4d shape: {X_train_4d.shape}')
print(f'X_val_4d shape:   {X_val_4d.shape}')
print(f'X_test_4d shape:  {X_test_4d.shape}')

In [ ]:
from tensorflow.keras import layers, models, regularizers

def build_model(input_shape, num_classes):
    """
    轻量级 Conv2D 模型，适合嵌入式部署。
    BN 与激活分离，Flatten 代替 GlobalAvgPool，量化友好。
    目标: 量化后 < 100KB
    """
    model = models.Sequential([
        # Block 1 — 时间轴卷积
        layers.Conv2D(8, (5, 1), padding='same', use_bias=False,
                      input_shape=input_shape),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 1)),

        # Block 2 — 时间轴卷积
        layers.Conv2D(16, (5, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 1)),

        # Block 3 — 时空联合卷积
        layers.Conv2D(24, (3, 3), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 1)),

        # Classifier head
        layers.Flatten(),
        layers.Dense(24, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model


model = build_model((WINDOW_SIZE, NUM_CHANNELS, 1), NUM_CLASSES)
model.summary()

## 11. 模型训练

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=25, restore_best_weights=True,
        verbose=1, mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'best_model.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1, mode='max')
]

print('Starting training...')
history = model.fit(
    X_train_4d, y_train,
    validation_data=(X_val_4d, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

## 12. 训练过程可视化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history['accuracy'], label='Train Accuracy')
ax2.plot(history.history['val_accuracy'], label='Val Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 13. 模型评估

In [ ]:
# 测试集评估
test_loss, test_acc = model.evaluate(X_test_4d, y_test, verbose=0)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')

# 详细分类报告
y_pred = model.predict(X_test_4d, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

print('\n' + '='*60)
print('Classification Report:')
print('='*60)
print(classification_report(y_test, y_pred_classes, target_names=LABEL_NAMES))

In [ ]:
# 混淆矩阵
cm = confusion_matrix(y_test, y_pred_classes)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (Test Accuracy: {test_acc*100:.1f}%)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 14. TFLite 转换与 INT8 量化

In [ ]:
def representative_dataset_gen():
    """为 INT8 量化提供代表性数据集"""
    # 从训练集中随机选 200 个样本用于校准
    indices = np.random.choice(len(X_train_4d), size=min(200, len(X_train_4d)), replace=False)
    for i in indices:
        sample = X_train_4d[i:i+1].astype(np.float32)
        yield [sample]


# 1. 转换为 TFLite (float32, 作为参考)
converter_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_f32 = converter_f32.convert()
f32_path = os.path.join(OUTPUT_DIR, 'model_float32.tflite')
with open(f32_path, 'wb') as f:
    f.write(tflite_f32)

# 2. 转换为 INT8 全量化
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8

tflite_int8 = converter_int8.convert()
int8_path = os.path.join(OUTPUT_DIR, 'model_int8.tflite')
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)

print(f'Float32 model size: {os.path.getsize(f32_path)/1024:.1f} KB')
print(f'INT8 model size:    {os.path.getsize(int8_path)/1024:.1f} KB')
print(f'Size reduction:     {(1 - os.path.getsize(int8_path)/os.path.getsize(f32_path))*100:.1f}%')

In [ ]:
# 验证 INT8 量化模型的精度
interpreter = tf.lite.Interpreter(model_path=int8_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('INT8 Model Input details:')
print(f'  Shape: {input_details[0]["shape"]}')
print(f'  Dtype: {input_details[0]["dtype"]}')
print(f'  Quantization (scale, zero_point): {input_details[0]["quantization"]}')

print('\nINT8 Model Output details:')
print(f'  Shape: {output_details[0]["shape"]}')
print(f'  Dtype: {output_details[0]["dtype"]}')
print(f'  Quantization (scale, zero_point): {output_details[0]["quantization"]}')

# 获取输入量化参数
input_scale = input_details[0]['quantization'][0]
input_zero_point = input_details[0]['quantization'][1]
output_scale = output_details[0]['quantization'][0]
output_zero_point = output_details[0]['quantization'][1]

# 在测试集上评估量化模型
correct = 0
total = len(X_test_4d)

for i in range(total):
    # 量化输入
    input_data = X_test_4d[i:i+1].astype(np.float32)
    input_quantized = (input_data / input_scale + input_zero_point).astype(np.int8)
    
    interpreter.set_tensor(input_details[0]['index'], input_quantized)
    interpreter.invoke()
    
    output_data = interpreter.get_tensor(output_details[0]['index'])
    pred = np.argmax(output_data)
    
    if pred == y_test[i]:
        correct += 1

int8_accuracy = correct / total
print(f'\nINT8 Quantized Model Test Accuracy: {int8_accuracy:.4f} ({int8_accuracy*100:.1f}%)')
print(f'Float32 Model Test Accuracy:         {test_acc:.4f} ({test_acc*100:.1f}%)')
print(f'Accuracy drop after quantization:    {(test_acc - int8_accuracy)*100:.2f}%')

## 15. 生成 model.h 和 model.c (C 字节数组)

In [ ]:
def tflite_to_c_array(tflite_path, var_name='g_model'):
    """将 .tflite 文件转换为 C 字节数组字符串"""
    with open(tflite_path, 'rb') as f:
        data = f.read()
    
    c_array_lines = []
    for i in range(0, len(data), 12):
        chunk = data[i:i+12]
        hex_str = ', '.join(f'0x{b:02x}' for b in chunk)
        c_array_lines.append(f'  {hex_str},')
    
    # 去掉最后一行的末尾逗号
    if c_array_lines:
        c_array_lines[-1] = c_array_lines[-1].rstrip(',')
    
    return c_array_lines, len(data)


c_lines, model_size = tflite_to_c_array(int8_path, 'g_model')

# ============ 生成 model.h ============
model_h_content = f"""/*
 * Movement Type Detection - TFLite Micro Model
 *
 * Auto-generated model header file.
 * Model: INT8 quantized Conv2D classifier
 * Input:  {WINDOW_SIZE} timesteps x {NUM_CHANNELS} channels (100x6x1), INT8
 * Output: {NUM_CLASSES} classes [{', '.join(LABEL_NAMES)}]
 * Size:   {model_size} bytes ({model_size/1024:.1f} KB)
 *
 * Sampling Rate:  {SAMPLE_RATE} Hz
 * Window Size:    {WINDOW_SIZE} samples ({WINDOW_SEC}s)
 * Window Step:    {STEP_SIZE} samples ({STEP_SIZE/SAMPLE_RATE}s)
 */

#ifndef MODEL_H_
#define MODEL_H_

#ifdef __cplusplus
extern "C" {{
#endif

/* Model data (INT8 quantized TFLite FlatBuffer) */
extern const unsigned char g_model[];
extern const unsigned int  g_model_len;

/* Model metadata constants */
#define MODEL_INPUT_SAMPLES   {WINDOW_SIZE}
#define MODEL_INPUT_CHANNELS  {NUM_CHANNELS}
#define MODEL_NUM_CLASSES     {NUM_CLASSES}
#define MODEL_SAMPLE_RATE     {SAMPLE_RATE}
#define MODEL_WINDOW_STEP     {STEP_SIZE}

/* Class label indices */
#define CLASS_STATIONARY  0
#define CLASS_SHAKING     1
#define CLASS_CIRCLE      2

/* Class label strings */
static const char* const g_class_names[MODEL_NUM_CLASSES] = {{
    "stationary",
    "shaking",
    "circle"
}};

/* Input quantization parameters (for converting float to int8) */
#define INPUT_SCALE       {input_scale}f
#define INPUT_ZERO_POINT  {input_zero_point}

/* Output quantization parameters (for converting int8 to float) */
#define OUTPUT_SCALE       {output_scale}f
#define OUTPUT_ZERO_POINT  {output_zero_point}

#ifdef __cplusplus
}}
#endif

#endif  /* MODEL_H_ */
"""

model_h_path = os.path.join(OUTPUT_DIR, 'model.h')
with open(model_h_path, 'w') as f:
    f.write(model_h_content)

print(f'Generated: {model_h_path}')
print(f'Model size: {model_size} bytes ({model_size/1024:.1f} KB)')

In [ ]:
# ============ 生成 model.c ============
c_array_str = '\n'.join(c_lines)

model_c_content = f"""/*
 * Movement Type Detection - TFLite Micro Model Data
 *
 * Auto-generated model data file.
 * Total size: {model_size} bytes ({model_size/1024:.1f} KB)
 */

#include "model.h"

#if defined(__GNUC__)
__attribute__((aligned(16)))
#endif
const unsigned char g_model[] = {{
{c_array_str}
}};

const unsigned int g_model_len = {model_size};
"""

model_c_path = os.path.join(OUTPUT_DIR, 'model.c')
with open(model_c_path, 'w') as f:
    f.write(model_c_content)

print(f'Generated: {model_c_path}')
print(f'model.c file size: {os.path.getsize(model_c_path)/1024:.1f} KB')

## 16. 生成预处理代码 (C 语言)

生成嵌入式端所需的预处理逻辑：
- 滑动窗口缓冲区管理
- Min-Max 归一化
- Float -> INT8 量化

In [ ]:
# ============ 生成 preprocessing.h ============

min_vals_str = ', '.join(f'{v:.6f}f' for v in channel_min)
max_vals_str = ', '.join(f'{v:.6f}f' for v in channel_max)

preprocessing_h_content = f"""/*
 * Movement Type Detection - Preprocessing for Embedded Deployment
 *
 * Handles:
 *   1. Sliding window buffer management
 *   2. Min-Max normalization
 *   3. Float -> INT8 quantization for model input
 *
 * Usage:
 *   1. Call preprocess_init() once at startup
 *   2. For each new IMU sample, call preprocess_add_sample()
 *   3. When preprocess_add_sample() returns true, the input buffer
 *      is ready for inference at preprocess_get_model_input()
 */

#ifndef PREPROCESSING_H_
#define PREPROCESSING_H_

#include <stdint.h>
#include <stdbool.h>
#include "model.h"

#ifdef __cplusplus
extern "C" {{
#endif

/* ---- Configuration ---- */
#define PREPROC_WINDOW_SIZE    MODEL_INPUT_SAMPLES    /* {WINDOW_SIZE} */
#define PREPROC_NUM_CHANNELS   MODEL_INPUT_CHANNELS   /* {NUM_CHANNELS} */
#define PREPROC_WINDOW_STEP    MODEL_WINDOW_STEP      /* {STEP_SIZE} */
#define PREPROC_BUFFER_LEN     (PREPROC_WINDOW_SIZE * PREPROC_NUM_CHANNELS)

/* ---- Normalization parameters (from training data) ---- */
/* Channel order: Accel_X, Accel_Y, Accel_Z, Gyro_X, Gyro_Y, Gyro_Z */
static const float CHANNEL_MIN[PREPROC_NUM_CHANNELS] = {{ {min_vals_str} }};
static const float CHANNEL_MAX[PREPROC_NUM_CHANNELS] = {{ {max_vals_str} }};

/**
 * Initialize the preprocessing module.
 * Must be called once before any calls to preprocess_add_sample().
 */
void preprocess_init(void);

/**
 * Add a single IMU sample (6 float values: ax, ay, az, gx, gy, gz).
 *
 * @param sample  Array of {NUM_CHANNELS} float values [accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z]
 * @return true   if the window buffer is full and ready for inference
 *         false  if more samples are needed
 */
bool preprocess_add_sample(const float sample[PREPROC_NUM_CHANNELS]);

/**
 * Get pointer to the quantized INT8 model input buffer.
 * Valid only after preprocess_add_sample() returns true.
 *
 * @return pointer to int8_t array of size PREPROC_BUFFER_LEN
 */
const int8_t* preprocess_get_model_input(void);

/**
 * Decode quantized INT8 output to float probabilities.
 *
 * @param raw_output   INT8 output from the model (MODEL_NUM_CLASSES values)
 * @param probs        Output float array to store probabilities
 */
void preprocess_decode_output(const int8_t raw_output[MODEL_NUM_CLASSES],
                              float probs[MODEL_NUM_CLASSES]);

/**
 * Get the predicted class index from float probabilities.
 *
 * @param probs  Float array of MODEL_NUM_CLASSES probabilities
 * @return index of the class with highest probability
 */
int preprocess_argmax(const float probs[MODEL_NUM_CLASSES]);

#ifdef __cplusplus
}}
#endif

#endif  /* PREPROCESSING_H_ */
"""

preprocessing_h_path = os.path.join(OUTPUT_DIR, 'preprocessing.h')
with open(preprocessing_h_path, 'w') as f:
    f.write(preprocessing_h_content)

print(f'Generated: {preprocessing_h_path}')

In [ ]:
# ============ 生成 preprocessing.c ============

preprocessing_c_content = f"""/*
 * Movement Type Detection - Preprocessing Implementation
 */

#include "preprocessing.h"
#include <string.h>

/* ---- Internal state ---- */
static float    s_window_buf[PREPROC_WINDOW_SIZE][PREPROC_NUM_CHANNELS];
static int8_t   s_model_input[PREPROC_BUFFER_LEN];
static int      s_write_idx;     /* next write position in window */
static int      s_samples_since_inference;  /* samples since last inference */
static bool     s_buffer_full;   /* whether we have a full window */

/* ---- Implementation ---- */

void preprocess_init(void)
{{
    memset(s_window_buf, 0, sizeof(s_window_buf));
    memset(s_model_input, 0, sizeof(s_model_input));
    s_write_idx = 0;
    s_samples_since_inference = 0;
    s_buffer_full = false;
}}

/**
 * Normalize a single value: (val - min) / (max - min) => [0, 1]
 */
static float normalize_channel(float val, int ch)
{{
    float range = CHANNEL_MAX[ch] - CHANNEL_MIN[ch];
    if (range < 1e-8f) return 0.0f;
    float norm = (val - CHANNEL_MIN[ch]) / range;
    /* Clamp to [0, 1] */
    if (norm < 0.0f) norm = 0.0f;
    if (norm > 1.0f) norm = 1.0f;
    return norm;
}}

/**
 * Convert a float [0, 1] value to INT8 using quantization params.
 */
static int8_t float_to_int8(float val)
{{
    int32_t quantized = (int32_t)(val / INPUT_SCALE) + INPUT_ZERO_POINT;
    if (quantized < -128) quantized = -128;
    if (quantized >  127) quantized = 127;
    return (int8_t)quantized;
}}

bool preprocess_add_sample(const float sample[PREPROC_NUM_CHANNELS])
{{
    /* Store raw sample in circular window buffer */
    for (int ch = 0; ch < PREPROC_NUM_CHANNELS; ch++) {{
        s_window_buf[s_write_idx][ch] = sample[ch];
    }}
    
    s_write_idx++;
    s_samples_since_inference++;
    
    /* Check if we have a full window */
    if (s_write_idx >= PREPROC_WINDOW_SIZE) {{
        s_buffer_full = true;
    }}
    
    /* Trigger inference when:
     *   1. Buffer is full (first time: after WINDOW_SIZE samples)
     *   2. Enough new samples since last inference (WINDOW_STEP samples)
     */
    if (s_buffer_full && s_samples_since_inference >= PREPROC_WINDOW_STEP) {{
        /* Normalize and quantize the current window */
        int read_start = s_write_idx - PREPROC_WINDOW_SIZE;
        if (read_start < 0) read_start = 0;
        
        int out_idx = 0;
        for (int t = read_start; t < read_start + PREPROC_WINDOW_SIZE; t++) {{
            for (int ch = 0; ch < PREPROC_NUM_CHANNELS; ch++) {{
                float norm_val = normalize_channel(s_window_buf[t][ch], ch);
                s_model_input[out_idx++] = float_to_int8(norm_val);
            }}
        }}
        
        s_samples_since_inference = 0;
        
        /* Shift buffer: move the last STEP samples to the beginning */
        if (s_write_idx >= PREPROC_WINDOW_SIZE) {{
            int keep_from = PREPROC_WINDOW_STEP;
            memmove(s_window_buf[0], s_window_buf[keep_from],
                    (PREPROC_WINDOW_SIZE - PREPROC_WINDOW_STEP) * PREPROC_NUM_CHANNELS * sizeof(float));
            s_write_idx = PREPROC_WINDOW_SIZE - PREPROC_WINDOW_STEP;
        }}
        
        return true;  /* Ready for inference */
    }}
    
    return false;  /* Need more samples */
}}

const int8_t* preprocess_get_model_input(void)
{{
    return s_model_input;
}}

void preprocess_decode_output(const int8_t raw_output[MODEL_NUM_CLASSES],
                              float probs[MODEL_NUM_CLASSES])
{{
    for (int i = 0; i < MODEL_NUM_CLASSES; i++) {{
        probs[i] = ((float)raw_output[i] - OUTPUT_ZERO_POINT) * OUTPUT_SCALE;
        if (probs[i] < 0.0f) probs[i] = 0.0f;
        if (probs[i] > 1.0f) probs[i] = 1.0f;
    }}
}}

int preprocess_argmax(const float probs[MODEL_NUM_CLASSES])
{{
    int best_idx = 0;
    float best_val = probs[0];
    for (int i = 1; i < MODEL_NUM_CLASSES; i++) {{
        if (probs[i] > best_val) {{
            best_val = probs[i];
            best_idx = i;
        }}
    }}
    return best_idx;
}}
"""

preprocessing_c_path = os.path.join(OUTPUT_DIR, 'preprocessing.c')
with open(preprocessing_c_path, 'w') as f:
    f.write(preprocessing_c_content)

print(f'Generated: {preprocessing_c_path}')

## 17. 最终输出汇总

In [ ]:
import json

# 保存归一化参数为 JSON（方便后续使用）
norm_params_path = os.path.join(OUTPUT_DIR, 'normalization_params.json')
with open(norm_params_path, 'w') as f:
    json.dump(norm_params, f, indent=2)

# 保存训练报告
report = {
    'model_architecture': 'Conv2D (3 blocks) + GlobalAvgPool + Dense',
    'input_shape': [WINDOW_SIZE, NUM_CHANNELS, 1],
    'output_classes': LABEL_NAMES,
    'sample_rate_hz': SAMPLE_RATE,
    'window_size_samples': WINDOW_SIZE,
    'window_size_seconds': WINDOW_SEC,
    'window_step_samples': STEP_SIZE,
    'total_windows': int(len(X_all)),
    'train_windows': int(len(X_train)),
    'val_windows': int(len(X_val)),
    'test_windows': int(len(X_test)),
    'float32_test_accuracy': float(test_acc),
    'int8_test_accuracy': float(int8_accuracy),
    'float32_model_size_kb': round(os.path.getsize(f32_path) / 1024, 1),
    'int8_model_size_kb': round(os.path.getsize(int8_path) / 1024, 1),
    'input_quantization': {
        'scale': float(input_scale),
        'zero_point': int(input_zero_point)
    },
    'output_quantization': {
        'scale': float(output_scale),
        'zero_point': int(output_zero_point)
    },
    'normalization': norm_params
}

report_path = os.path.join(OUTPUT_DIR, 'training_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

# 汇总输出
print('=' * 65)
print('       MOVEMENT TYPE DETECTION — OUTPUT SUMMARY')
print('=' * 65)
print()
print(f'  Output directory: {OUTPUT_DIR}')
print()
print('  Generated files:')

output_files = [
    ('model.h',                'Model header (C byte array declaration)'),
    ('model.c',                'Model data   (C byte array definition)'),
    ('preprocessing.h',       'Preprocessing header (sliding window + normalization)'),
    ('preprocessing.c',       'Preprocessing implementation'),
    ('model_int8.tflite',     'INT8 quantized TFLite model'),
    ('model_float32.tflite',  'Float32 TFLite model (reference)'),
    ('best_model.keras',      'Best Keras model checkpoint'),
    ('normalization_params.json', 'Normalization parameters'),
    ('training_report.json',  'Full training report'),
    ('class_distribution.png','Class distribution chart'),
    ('sample_windows.png',    'Sample window visualization'),
    ('training_curves.png',   'Training curves plot'),
    ('confusion_matrix.png',  'Confusion matrix plot'),
]

for fname, desc in output_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f'    ✅ {fname:<32s}  {size_kb:>8.1f} KB  — {desc}')
    else:
        print(f'    ❌ {fname:<32s}  (missing)     — {desc}')

print()
print(f'  Float32 Test Accuracy: {test_acc*100:.1f}%')
print(f'  INT8 Test Accuracy:    {int8_accuracy*100:.1f}%')
print(f'  INT8 Model Size:       {os.path.getsize(int8_path)/1024:.1f} KB')
print()
print('  To deploy on your PSoC-Edge board:')
print('    1. Copy model.h, model.c, preprocessing.h, preprocessing.c to your project')
print('    2. Include TFLite Micro runtime library')
print('    3. Feed IMU samples at 50Hz to preprocess_add_sample()')
print('    4. When it returns true, run inference with preprocess_get_model_input()')
print('=' * 65)